# <div align="center"> Statistical Learning </div>
# <div align="center"> Machine Learning and Econometrics </div>
## <div align="center"> ECO 4971/6971 </div>
#### <div align="center">Class 14 — Social Biases and Prediction</div>
<div align="center"> Jonathan Holmes (he/him)</div>

# Road Map

We have now covered the full statistical learning toolkit, from linear regression to deep learning to clustering. This final class steps back from the mechanics and asks two bigger questions: *what can go wrong when these models are deployed in society*, and *where is the field heading next?*

# Part 1: The Machine Learning Framework in One Expression

Every method in this course — linear regression, logistic regression, KNN, trees, random forests, boosting, neural networks, ridge, lasso, PCA, K-means — fits into a single template. It is worth seeing them all together before we talk about what can go wrong.

We start from the same decomposition we wrote on day one,

$$Y = f(X) + \varepsilon,$$

where $f$ is the true (unknown) relationship between features and outcome and $\varepsilon$ is irreducible noise. Machine learning is the business of choosing a family of candidate functions, estimating one from data, and measuring how close the estimate $\hat{f}$ gets to $f$.

## The unified optimisation problem

Almost every supervised method we have studied can be written as

$$\min_{f \in \mathcal{F}} \; \underbrace{\sum_{i=1}^n L\big(f(x_i),\, y_i\big)}_{\text{in-sample loss}} \;+\; \underbrace{\lambda \sum_{k=1}^K P(\beta_k)}_{\text{complexity penalty}}.$$

The analyst makes three choices:

1. **The function class $\mathcal{F}$** — what shape can $\hat{f}$ take?
2. **The loss $L$** — how do we score a prediction against the truth?
3. **The penalty $P$ and its strength $\lambda$** — how hard do we fight overfitting?

Different choices give different methods. Everything else is implementation detail.

## Choice 1: the function class

The function class determines what kinds of relationships $\hat{f}$ can capture. We have studied several:

- **Linear models** — $f(X) = \beta_0 + \beta_1 X_1 + \dots + \beta_p X_p$. Simple, interpretable, low variance.
- **Logistic models** — linear in the log-odds, for classification.
- **K-nearest neighbours** — no parametric form at all; predictions are local averages.
- **Decision trees** — piecewise-constant functions defined by recursive splits.
- **Random forests and boosted trees** — ensembles of many trees, averaged or stacked.
- **Neural networks** — compositions of linear layers and non-linearities; arbitrarily flexible with enough units.

No single function class wins on every task. Flexible classes (neural nets, deep trees) have low bias but high variance; rigid classes (linear models) have the opposite. Which one you prefer depends on the signal-to-noise ratio, the sample size, and how much you value interpretability.

## Choice 2: the loss function

The loss measures the cost of a bad prediction. We have used:

- **Mean squared error** — $L(\hat{f}(x_i), y_i) = \tfrac{1}{n}\sum_i (\hat{f}(x_i) - y_i)^2$. Standard for regression.
- **Mean absolute error** — $\tfrac{1}{n}\sum_i |\hat{f}(x_i) - y_i|$. More robust to outliers.
- **Cross-entropy (log loss)** — the natural loss for classification, closely related to the logistic likelihood.

Other losses exist — Huber loss, hinge loss, quantile losses — and the choice matters. MSE punishes large errors quadratically, which is wrong if your application tolerates a few bad predictions but not systematically biased ones (or vice versa).

## Choice 3: the complexity penalty

Without a penalty, a flexible model will memorise the training data and generalise poorly. The penalty term $\lambda \sum_k P(\beta_k)$ pushes the coefficients toward zero. We have seen three regimes:

- **No penalty** ($\lambda = 0$) — OLS, standard logistic regression, untrained neural networks.
- **Lasso** — $P(\beta_k) = |\beta_k|$. Shrinks small coefficients all the way to zero, performing feature selection.
- **Ridge** — $P(\beta_k) = \beta_k^2$. Shrinks coefficients smoothly without zeroing them out.

$\lambda$ itself is a hyperparameter, usually chosen by cross-validation. We generally call penalisation of this form **regularisation**, and it shows up everywhere — from lasso regression to the weight decay term inside a Keras neural network.

## Mix and match

Because the three choices are independent, you are free to combine them however your application demands. We showed the canonical pairings (linear + lasso = lasso regression; tree + boosting = XGBoost), but the template does not restrict you to those. A neural network with lasso weight decay is perfectly sensible. So is ridge-regularised logistic regression, or an ensemble of KNN models with different $k$.

The point is that methods are not disjoint recipes — they are points in a three-dimensional design space. Once you see the space, inventing a method becomes a matter of navigating it.

## Measuring how good $\hat{f}$ is

Finally, we need a way to compare candidate models. We have used:

- **Mean squared error and mean absolute error** on a held-out test set.
- **Adjusted $R^2$, AIC, BIC** — penalised in-sample fit measures.
- **$k$-fold cross-validation** — the workhorse for honest out-of-sample estimates.
- **Classification-specific metrics** — accuracy, precision, recall, F1, ROC/AUC, and the confusion matrix.

No single number tells the whole story. In Part 2 we will see cases where a model with high overall accuracy performs disastrously on a specific subgroup — a reminder that averaged metrics can hide the failures that matter most.

## Trying methods automatically

Because no function class dominates, a reasonable default when starting a new problem is to try many at once and compare. The Python library [PyCaret](https://pycaret.org/) does exactly this: fit a dozen candidate models, cross-validate them, and rank by a metric of your choice. Its [tutorial notebooks](https://github.com/pycaret/pycaret/tree/master/tutorials) are a good starting point. This is a reasonable way to get a baseline; it is *not* a substitute for understanding the methods, because PyCaret cannot diagnose the bias problems we turn to next.

# Part 2: Social Biases and Prediction

Everything we have done so far has optimised for accuracy. That is the right objective on a Kaggle leaderboard. When the model is deployed on human beings — to screen resumes, allocate medical care, set bail, or generate faces — accuracy is not enough. A model can be accurate on average and still cause serious harm to a specific group.

This part covers three ways bias enters a trained model, each with a concrete case study, and ends with what researchers are doing about it.

1. **Sample-induced bias** — the training data do not represent the population the model is deployed on.
2. **Society-induced bias** — the data faithfully reflect a world that is itself biased, and the model learns that bias.
3. **Missing features** — the variables we measure are not the variables that actually matter, and the model substitutes correlated proxies.

## Sample-induced bias: the face depixelizer

In 2020 a research group released a neural network called PULSE that upsampled low-resolution face photos. Given an 8×8-pixel input, the network output a plausible 32×32 face — one pixel in became four pixels out, chosen to look like a real high-resolution face.

Formally the network learns a function

$$\mathbf{Y} = f(\mathbf{X}),$$

where $\mathbf{X}$ is the low-resolution image and $\mathbf{Y}$ is the reconstructed high-resolution image. As with any supervised method, it was trained by feeding in downsampled versions of real photos and using the original photos as targets. The network learned the mapping that minimised reconstruction loss on the training set.

![Neural network upsampling diagram](https://4.img-dpreview.com/files/p/E~TS590x0~articles/4871415337/googlebrain.jpeg)

## A famous test case

A Twitter user fed the network a low-resolution photo of a well-known politician. See if you recognise him.

![Low-resolution input photo](https://www.dropbox.com/scl/fi/res6crvths6waef0ii70a/Obama.png?rlkey=j83u7jtjtyg5mfte1q15wgctv&dl=1)

To a human viewer the subject is obvious: this is a pixelated Barack Obama. We recognise him because we have seen his face many times and our visual system fills in the missing detail from memory. The neural network does not have that option. It has only the weights it learned from its training data, and it returns the high-resolution face that best matches those weights given the low-resolution input. Here is what it produced:

![Depixelizer output](https://cdn.vox-cdn.com/thumbor/MXX-mZqWLQZW8Fdx1ilcFEHR8Wk=/55x85:768x536/1820x1213/filters:focal(336x236:464x364):format(webp)/cdn.vox-cdn.com/uploads/chorus_image/image/66972412/face_depixelizer_obama.0.jpg)

The network returned the face of a white man. The same pattern showed up across many examples posted at the time:

![Additional depixelizer examples 1](https://pbs.twimg.com/media/Ea-8T2NXkAEfH6y?format=png&name=900x900)

![Additional depixelizer examples 2](https://pbs.twimg.com/media/Ea_AGceXYAYg4KT?format=jpg&name=medium)

Is the model *wrong*? Strictly speaking, no — it returned a perfectly plausible face consistent with the low-resolution input. But it consistently returned white faces even when the subject was not white. The reason is simple: the training set (FFHQ, a standard face dataset scraped from Flickr) is overwhelmingly composed of white faces. The network learned that, in the absence of strong pixel-level evidence, a face that looks "like the training distribution" is a white face.

This is **sample-induced bias**: the training sample is not representative of the population the model is asked to make predictions about.

### Mitigation

The cleanest fix is at the data stage: construct a training sample that is representative of the population of deployment. This is easier said than done — representative samples are expensive, and for some populations the underlying images or records simply do not exist in comparable quantities. But it is the most durable solution, because the problem originates in the data rather than in the loss or the architecture.

## Society-induced bias: when the data reflect a biased world

Sample-induced bias is at least conceptually easy: collect better data. The next class of problems is harder, because the data collection is unbiased in a narrow technical sense and still produces a biased model. The issue is that the world we are measuring is itself unequal.

### Example 1: Amazon's resume screener

Amazon built an internal machine learning tool to screen job applications. The training data were resumes from previous applicants, labelled by whether the person was hired. The algorithm's job was to predict which of today's resumes looked like resumes that had been hired in the past.

Most past hires were men. The model learned to downweight signals associated with women — attendance at all-women's colleges, membership in women's organisations, even the word "women's" as in "women's chess club captain". Amazon shut the project down when they could not fix it. The data were accurate. The model was accurate at its stated objective (*predict past hiring decisions*). It was also unusable because past hiring decisions were themselves biased.

### Example 2: health spending as a proxy for health

A widely used risk-prediction tool in the US healthcare system was designed to identify the sickest patients who would most benefit from an enhanced care program. The label used to train the model was *past healthcare spending* — a natural proxy for health, since sicker patients generate more claims. [Obermeyer et al. (2019)](https://www.science.org/doi/10.1126/science.aax2342) audited the model and found it under-identified Black patients at every level of sickness.

The reason: at the same level of disease burden, Black patients historically received less care, so their spending was lower. The model learned the mapping from covariates to *spending*, not from covariates to *sickness*. Because the proxy was racially biased, the predictions were too.

### Example 3: chatbots trained on the internet

Large language models are trained on hundreds of billions of words scraped from the public internet. The internet contains a lot of prejudice — racist, sexist, and otherwise abusive content — alongside useful information. Early chatbots (Microsoft's Tay in 2016 is the canonical example) absorbed the prejudice and reproduced it. Modern LLMs still do so under the right prompts; the mitigation (reinforcement learning from human feedback, content filters) is a layer on top, not an elimination of the underlying pattern.

### Mitigation

Society-induced bias is genuinely hard because the signal and the bias live in the same variables. Three broad approaches are in use, usually in combination:

- **Curate or reweight the training data** where the bias can be identified — drop obviously toxic content, upweight under-represented groups.
- **Censor the output** — filter generated text, reject classifications that fall below fairness thresholds. ChatGPT-style safety layers are examples.
- **Audit the predictions by subgroup** — compute accuracy, false-positive rate, and false-negative rate separately for each protected group, and adjust the model (or the decision threshold) until the gaps are acceptable.

None of these is a silver bullet. The deeper point is that an algorithm cannot be "fair" in isolation — fairness is a property of the *deployment*, which includes who is measured, which label is used, and what decision the prediction feeds into.

## Case study: predicting recidivism (COMPAS)

Several US states use machine learning to help judges set bail and sentences. The goal is to predict a defendant's risk of committing a future crime — a classification task. In principle, an algorithm could remove the subjectivity and inconsistency of individual judges.

The for-profit company Northpointe (now Equivant) built the most widely used tool, COMPAS. In 2016, ProPublica audited it on a sample of Florida defendants. Their findings became one of the most-cited results in the algorithmic fairness literature.

Recall the confusion matrix for a classification task:

| Prediction \ Reality | Did not re-offend | Re-offended |
| --- | --- | --- |
| **Predicted low-risk** | True negative | False negative |
| **Predicted high-risk** | False positive | True positive |

ProPublica's [investigation](https://www.propublica.org/article/machine-bias-risk-assessments-in-criminal-sentencing) found:

- The algorithm was somewhat more accurate than a coin flip. Of those labelled high-risk, 61% were arrested for a subsequent crime within two years.
- **Black defendants were falsely labelled as high-risk at almost twice the rate of white defendants** (false positive rate).
- **White defendants were falsely labelled as low-risk more often than Black defendants** (false negative rate).

In other words, the errors were not distributed evenly. The cost of a false positive — being denied bail, serving a longer sentence — landed disproportionately on Black defendants.

### Why race bias without a race variable?

Crucially, race was *not* an input to the COMPAS model. How can a race-blind model produce race-biased predictions?

The answer is **proxy variables**. Many features the model does use — ZIP code, prior arrests, family criminal history, employment status — are correlated with race because of historical segregation and policing patterns. A model denied the race variable will happily use any combination of correlated features to recover the same information. Removing the protected attribute from the feature matrix does not remove its influence; it only hides it from the analyst.

This is an example of the third mechanism: the feature set is *incomplete*. The variables that actually drive reoffending (opportunity, community, mental health, substance use context) are only partially captured, and the model fills the gap with whatever correlated signal it can find.

## Eliminating bias: current research

Algorithmic fairness is an active area of research at the intersection of computer science, statistics, and economics. A few threads worth knowing about:

- **Causal approaches.** Econometric tools (instrumental variables, regression discontinuity, difference-in-differences) are being combined with ML to distinguish *predictive* relationships from *causal* ones. A model that predicts who will reoffend is not the same as a model of what *causes* reoffending.
- **Formal fairness definitions.** Demographic parity, equal opportunity, calibration by group — each is a precise statistical criterion. An important result (Chouldechova 2017, Kleinberg–Mullainathan–Raghavan 2016) is that these criteria are mutually incompatible when base rates differ across groups. "Fair" is not a single well-defined target.
- **Practitioner guidance.** University of Chicago's [Algorithmic Bias Playbook](https://www.chicagobooth.edu/-/media/project/chicago-booth/centers/caai/docs/algorithmic-bias-playbook-june-2021) is a practical starting point for auditing a deployed model.

The short version: there is no "unbias" button. Building a model that is fair in deployment requires understanding the data-generating process, the decision it feeds into, and the population it affects — economics and statistics, not just ML.

# Part 3: The Pace of Machine Learning Progress

The methods in this course cover roughly eighty years of statistics — linear regression dates from Gauss in the early 1800s, logistic regression from the 1940s, trees and boosting from the 1980s and 1990s, deep learning from the 2010s. What has changed in the last decade is not the underlying math but the *speed of progress on practical tasks*.

### Compute is growing exponentially

Training compute for frontier models — measured in floating-point operations — has roughly doubled every six months since 2012. That is far faster than Moore's law (which doubled every 18–24 months for transistor counts). The doubling is driven by three compounding factors: larger chips, larger clusters of chips, and longer training runs. A 2012 image classifier used about $10^{15}$ FLOPs to train. A 2025 frontier LLM uses roughly $10^{26}$ — eleven orders of magnitude more.

### Benchmarks are saturating faster than they can be built

Standardised benchmarks are how the field measures progress. In the 2010s, a new benchmark typically lasted five to ten years before models matched human performance. That window has collapsed. MMLU (a broad knowledge test released in 2020) was near-saturated by 2024. GSM8K (grade-school math, released 2021) was solved within two years. Harder successors like MATH, GPQA, and SWE-bench have followed the same arc: released, attacked, solved in months rather than years.

When you hear "AI has plateaued," check the benchmark. Specific capabilities plateau; new ones keep appearing just as fast.

## Large language models and general-purpose AI

Until about 2020, ML systems were narrow: a model that labelled images could not translate text, and a model that translated text could not play chess. Large language models blurred that line. A single model can now, with appropriate prompting, write code, summarise medical records, answer exam questions, translate between languages, and carry on a conversation.

The underlying method is the same neural network idea from Class 12, scaled up by several orders of magnitude: transformer architectures with hundreds of billions of parameters, trained on trillions of tokens of text. None of the components is conceptually new. What is new is the empirical observation that performance continues to improve with scale — more data, more parameters, more compute — across a wide range of downstream tasks.

### From chatbots to agents

The frontier in 2026 is no longer chat. It is *agents*: models that can take actions in the world — browse the web, write and run code, send emails, call APIs — in pursuit of a goal specified in natural language. Claude Code, Cursor, and similar tools let a single developer (or a single model) produce code that used to take a small team. The same technology applied to office work, research, and customer service is reshaping what knowledge jobs look like.

These agents are still unreliable in ways that matter — they hallucinate, they get stuck in loops, they can be manipulated by adversarial inputs — and reasonable people disagree about how quickly these limitations will be resolved. But the *direction* is clear, and it is worth factoring into how you think about your own career.

# Part 4: Contemporary Concerns

The bias issues in Part 2 are well-studied and have a research community around them. Parts 3's pace of progress has created a set of newer concerns that are less settled. Reasonable, well-informed people disagree about how serious each is and what to do about them. Here is a tour of the main debates as they stand today.

## Misinformation and deepfakes at scale

Generative models can now produce photorealistic images, convincing audio, and plausible video of people doing or saying things they did not do. The 2024 election cycles in the US and India both featured fabricated robocalls and video clips of candidates. In March 2026, a financial-services firm in Hong Kong lost roughly US$25 million after an employee was deepfaked into a video call with a fake CFO.

The technical fix (cryptographic provenance signatures on real content, classifier-based detection of fakes) lags the generation side and probably always will — it is easier to produce a convincing fake than to detect one at scale. The more durable solutions are institutional: verified communication channels, authenticated sources, slower news cycles. Those are social, not statistical, problems.

## Labour market disruption

The classical concern about automation is that it displaces workers in routine tasks. What is different about ML is that the *non-routine cognitive* tasks — writing, analysis, coding, legal drafting, medical diagnosis — are increasingly automatable. These are exactly the jobs a university degree prepares people for.

Early evidence (2023–2025) suggests two patterns:

- **Within-occupation productivity gains** are large — customer-service agents, programmers, and writers using AI tools produce roughly 20–60% more output per hour than unassisted peers, with the largest gains accruing to less-experienced workers.
- **Between-occupation displacement** is starting to appear in specific niches — entry-level copywriting, first-draft translation, junior legal research.

How this plays out depends on the *elasticity of demand* for the affected output. If AI makes writing cheaper and people read more, the total number of writers employed may grow. If demand is fixed, fewer writers are needed. The honest answer is that we do not yet know, and the answer is likely to be different for different occupations.

## Copyright, consent, and creative work

Most frontier models are trained on datasets that include copyrighted text, images, and code scraped from the public internet. The creators of that material were not asked and were not compensated. A wave of lawsuits — *NYT v. OpenAI*, *Getty Images v. Stability AI*, class actions by authors and artists — is working through the courts, and the outcomes will shape the economics of the field.

Two questions are genuinely hard:

- **Is training fair use?** US copyright doctrine turns on whether the use is "transformative." Reading a million novels to learn how prose works is arguably transformative. Producing a near-copy of a specific novel on demand is not. Where the line falls is what the courts are deciding.
- **What do creators deserve?** Even if training is legally permitted, the livelihoods of writers, artists, and musicians are affected by tools built on their work. Questions of compensation and consent are live, and the answers will be political as much as legal.

## Concentration of economic power

Training a frontier model in 2026 costs hundreds of millions to billions of dollars in compute. The companies that can afford it can be counted on two hands. That concentration is unusual in technology — a 1990s startup could write software on a laptop and reach global scale — and it has consequences.

- **Pricing power.** A small number of labs control the most capable models. They set the prices developers pay, which flow through to the prices consumers pay.
- **Research gatekeeping.** The most-cited ML research increasingly comes from industrial labs rather than universities, because academia cannot afford the compute.
- **Policy capture.** The same firms are the main source of technical expertise for governments writing AI regulation, which creates obvious conflict-of-interest concerns.

Open-weight models (Llama, Mistral, DeepSeek, and others) push against this concentration by releasing competitive models for free. Whether they stay close to the frontier depends on whether scaling continues to require budgets only the largest firms can sustain.

## Alignment, safety, and oversight

As models become more capable, a set of concerns that used to be speculative becomes concrete. *Alignment* is the problem of making sure a model does what its operator intends, rather than what it was literally trained to optimise. Three specific worries:

- **Reward hacking.** A model trained with a proxy objective (helpfulness ratings, click-through, task completion) may learn to game the proxy rather than satisfy the underlying intent. Classic reward-hacking is already documented in chatbots trained with reinforcement learning that flatter users rather than inform them.
- **Deception.** More-capable models can, under the right training signals, learn to behave differently when they believe they are being evaluated than when they believe they are deployed. Anthropic, OpenAI, and DeepMind all now publish pre-deployment evaluation suites to catch this; those suites are themselves imperfect.
- **Agent oversight.** An agent that can take actions in the world can cause more harm than one that only produces text. The safe deployment of agents is currently an engineering problem: sandboxing, rate limits, human-in-the-loop checkpoints. It may eventually be a research problem.

Whether these concerns justify slowing down frontier research, coordinating across labs, or treating AI development like nuclear weapons or biotech is the central policy debate of the field right now.

## Environmental and infrastructure costs

Training and serving large models consumes significant electricity and water. A single frontier training run can use tens of gigawatt-hours. Data centres housing these runs have become large enough to strain regional power grids; several US utilities have delayed coal-plant retirements to meet projected AI demand. Water usage — for evaporative cooling — is concentrated in a few geographic areas and competes with agriculture and municipal supply.

These are real costs, and they scale with usage rather than one-time training. Efficiency gains (better chips, better algorithms) are running against usage gains (more users, larger models) and the usage side is currently winning. How this resolves depends on the pace of chip efficiency improvements, the geographic distribution of compute, and whether clean-energy supply can keep up with demand.

## A closing thought

The course began with a simple equation, $Y = f(X) + \varepsilon$, and a single goal — estimate $\hat{f}$ well. That problem is now largely solved at the method level: we have a toolbox that, applied carefully, produces good predictions on most tabular and increasingly most unstructured data.

The unsolved problems are the ones this class covered. *What data should we train on?* *What are we optimising for?* *Who benefits and who is harmed by a prediction?* *How fast should we move?* These are not statistical questions. They are economic, legal, and political questions that sit on top of a statistical machine. You now have the statistical machine. The rest of your career is the harder part.

Thank you for taking the course.